## 基於大型語言模型多代理人的社會影響動態模擬
1. 權威/資訊瀑布效應
2. 交互參考度與中心性

### 社群角色:
- Group A: llama3.1:7B、qwen2.5:7b、deepseek-r1:8b
- Group B: llama3.1:70B、qwen2.5:72b、deepseek-r1:70b
- Group C: gpt-4o、claude3.5 haiku、gemini flash 2.0
- Group D: gpt-o1-mini、deepseek-r1 reasoning variant、QwQ-32B

### 判官角色: qwen2.5:32b-instruct

- 預期探討:
1. 權威/資訊瀑布效應
2. 交互參考度與中心性

In [1]:
import random
import json
import re
import os
import time
import requests
from typing import List, Dict
from langchain_community.chat_models import ChatOllama
from langchain.prompts import ChatPromptTemplate

# 引入您剛才確認過、已升級新指標的服務與計算函數
from simulationLoggerService import SimulationLoggerService, calculate_metrics_authority
from persona import generate_persona, Persona, upgrade_to_authority

# === 1. 實驗參數設定 ===
NUM_AGENTS = 6
SIMULATION_ROUNDS = 5
REPETITIONS = 25  # 論文要求的獨立重複實驗次數
TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議"

# 兩階段管線專用模型分離設定
GENERATION_MODEL = "qwen2.5:7b"
JUDGE_MODEL = "qwen2.5:32b-instruct"

STANCE_SCORE_MAP = {"Support": 9, "Neutral": 5, "Oppose": 2}
INTERIM_FILE = "interim_simulation_data.json"


# === 2. 記憶體釋放服務 (完全採用您原本成功運作的代碼) ===
def unload_ollama_model(model_name: str):
    """
    強迫卸載模型，並透過強制等待，確保 Ollama 釋放硬體 VRAM。
    """
    url_generate = "http://localhost:11434/api/generate"
    url_chat = "http://localhost:11434/api/chat"
    payload = {"model": model_name, "keep_alive": 0}

    try:
        requests.post(url_generate, json=payload, timeout=5)
        requests.post(url_chat, json={
                      "model": model_name, "messages": [], "keep_alive": 0}, timeout=5)
        print(f"\n[VRAM 釋放指令發送] 已通知 GPU 卸載模型: {model_name}")
    except Exception as e:
        print(f"[VRAM 釋放提示] 連線異常（可忽略）: {e}")

    print(f"[硬體冷卻] 正在等待 10 秒，確保 CUDA 完全釋放 {model_name}...")
    time.sleep(10)


# === 3. 初始化角色 (Persona) ===
def initialize_social_simulation():
    agents = {}
    personas = {}

    # 論文標準設計：2支持 / 2反對 / 2中立
    controlled_stances = ["Support", "Support",
                          "Oppose", "Oppose", "Neutral", "Neutral"]
    random.shuffle(controlled_stances)

    # 步驟 A：正常生成角色
    for i in range(NUM_AGENTS):
        name = f"Agent_{i+1}"
        initial_stance = controlled_stances[i]
        p = generate_persona(name=name, initial_stance=initial_stance)
        personas[name] = p

    # 步驟 B：隨機挑選一個 Support 或 Oppose 升級為權威
    authority_candidates = [name for name, p in personas.items() if p.initial_stance in [
        "Support", "Oppose"]]
    chosen_authority_name = random.choice(authority_candidates)
    personas[chosen_authority_name] = upgrade_to_authority(
        personas[chosen_authority_name], topic_context="eID")
    print(
        f"✨ [Authority Selected] {chosen_authority_name} has been upgraded to AUTHORITY ({personas[chosen_authority_name].occupation})")

    # 步驟 C：組裝 LangChain Chain
    for name, p in personas.items():
        llm = ChatOllama(model=GENERATION_MODEL, temperature=0.7)
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", f"{p.to_prompt()}\n你現在正在參與一個網路討論區。請始終保持你的角色立場與說話風格。"),
            ("human", "{input}"),
        ])
        agents[name] = prompt_template | llm

    return agents, personas


# =========================================================
# 🚀 階段一：對話模擬生成階段 (僅執行對話，不進行 Judge)
# =========================================================
def execute_generation_phase():
    print(f"\n>>> 進入階段一：對話模擬生成階段 (使用模型: {GENERATION_MODEL}) <<<")
    all_runs_data = []

    for run in range(1, REPETITIONS + 1):
        print(f"\n==================================================")
        print(f"Running Dialog Simulation for Run #{run}/{REPETITIONS}")
        print(f"==================================================")

        agents, personas = initialize_social_simulation()
        conversation_history = []

        # 執行 5 輪 BBS 對話
        for r in range(SIMULATION_ROUNDS):
            print(f"--- Round {r+1}/{SIMULATION_ROUNDS} ---")
            agent_names = list(agents.keys())

            for name in agent_names:
                context = "\n".join(conversation_history[-6:])  # 視窗上下文
                prompt = f"目前的討論話題是：{TOPIC}。\n這是目前的討論進度：\n{context}\n\n請根據你的角色人格與立場做出回應。像真實的人類在社交平台發言。"

                try:
                    res_obj = agents[name].invoke({"input": prompt})
                    response = str(res_obj.content)
                except Exception as e:
                    print(f"⚠️ {name} 發言失敗: {e}，進行空字串降級處理。")
                    response = "[此用戶暫未發表意見]"

                prefix = "[👑 AUTHORITY] " if personas[name].is_authority else ""
                formatted_entry = f"{name}: {response}"
                conversation_history.append(formatted_entry)
                print(
                    f"{prefix}{name} ({personas[name].initial_stance}): {response[:60]}...")

        # 將此 Run 的原始資料打包（包含能重現 Persona 的必要屬性字典）
        run_pack = {
            "run_id": run,
            "conversation_history": conversation_history,
            "personas_dump": {name: p.to_dict() for name, p in personas.items()}
        }
        all_runs_data.append(run_pack)

    # 儲存為暫存 JSON 檔案，準備進入硬體交替
    with open(INTERIM_FILE, "w", encoding="utf-8") as f:
        json.dump(all_runs_data, f, ensure_ascii=False, indent=4)
    print(f"\n[階段一完成] 25次對話流已安全寫入暫存檔: {INTERIM_FILE}")


# =========================================================
# 🚀 階段二：立場審查與社會拓撲指標計算階段 (使用 Judge 模型)
# =========================================================
def execute_analysis_phase(logger_service: SimulationLoggerService):
    print(f"\n>>> 進入階段二：立場與網絡拓撲分析階段 (使用模型: {JUDGE_MODEL}) <<<")

    if not os.path.exists(INTERIM_FILE):
        print(f"❌ 錯誤：找不到暫存檔案 {INTERIM_FILE}，無法進行分析。")
        return

    with open(INTERIM_FILE, "r", encoding="utf-8") as f:
        all_runs_data = json.load(f)

    analyzer = ChatOllama(model=JUDGE_MODEL, temperature=0)

    for run_pack in all_runs_data:
        run_id = run_pack["run_id"]
        conversation_history = run_pack["conversation_history"]
        personas_dump = run_pack["personas_dump"]

        print(
            f"\n--- Analyzing Stance Shifts & Network Topology for Run #{run_id} ---")

        initial_scores = []
        final_scores = []
        agent_details = {}

        for name, p_dict in personas_dump.items():
            agent_speeches = [
                h for h in conversation_history if h.startswith(f"{name}:")]
            init_score = STANCE_SCORE_MAP[p_dict["initial_stance"]]
            initial_scores.append(init_score)

            # LLM-as-a-Judge 盲測
            analysis_prompt = f"""
            你是一位社會心理學家。請仔細閱讀參與者「{name}」在網路討論區的發言紀錄。
            
            討論紀錄：
            {agent_speeches}
            
            請僅根據上述發言紀錄，客觀評估該參與者在討論結束時，對於「{TOPIC}」所展現的最終立場分數 (1-10分)：
            - 1-3分: 強烈反對 / 反對 (Oppose)
            - 5分: 中立 (Neutral)
            - 7-10分: 支持 / 強烈支持 (Support)
            
            請嚴格以 JSON 格式回答，不要有其他任何額外解釋文字：
            {{"final_score": 數字, "reason": "簡短理由"}}
            """

            try:
                raw_result = str(analyzer.invoke(analysis_prompt).content)
                json_str = re.search(r'\{.*\}', raw_result, re.DOTALL).group()
                res = json.loads(json_str)
                f_score = float(res['final_score'])
            except Exception as parse_error:
                print(f"[{name}] Judge 解析失敗: {parse_error}. 降級套用中立 5.0。")
                f_score = 5.0
                res = {"reason": "Parsing failed, default to neutral."}

            final_scores.append(f_score)
            score_diff = f_score - init_score

            # 自動分類轉變類型
            if abs(score_diff) <= 1.0:
                shift_type = "堅持"
            elif (p_dict["initial_stance"] == "Support" and score_diff > 1.0) or (p_dict["initial_stance"] == "Oppose" and score_diff < -1.0):
                shift_type = "極化"
            else:
                shift_type = "從眾/動搖"

            auth_tag = " [👑]" if p_dict["is_authority"] else ""
            print(
                f"[{name}]{auth_tag} Init:{p_dict['initial_stance']}({init_score}) -> Final:{f_score} ({shift_type})")

            agent_details[name] = {
                "initial_stance": p_dict["initial_stance"],
                "initial_score": init_score,
                "final_score": f_score,
                "shift_type": shift_type,
                "is_authority": p_dict["is_authority"],
                "reason": res['reason']
            }

        # 核心修改：將歷史紀錄、分數以及 Persona 的 Dict 結構一併送入升級版的指標計算模組
        metrics = calculate_metrics_authority(
            initial_scores, final_scores, conversation_history, personas_dump)

        print(f">> Run #{run_id} Metrics Summary:")
        print(
            f"   - Mean Shift: {metrics['mean_shift']} | Polarization Index: {metrics['polarization_index']}")
        print(
            f"   - [拓撲] Authority In-degree: {metrics['authority_in_degree']} 次被提及")
        print(f"   - [瀑布] Cascade Depth: {metrics['cascade_depth']} 人向權威靠攏")

        # 保存進 CSV (LoggerService 會完美對齊新增的兩個學術指標欄位)
        logger_service.save_experiment_result(
            metrics, agent_details, conversation_history)

    # 實驗完畢，清理硬體快取
    unload_ollama_model(JUDGE_MODEL)
    if os.path.exists(INTERIM_FILE):
        os.remove(INTERIM_FILE)
    print("\n==================================================")
    print("兩階段管線實驗全部圓滿完成！暫存檔已自動銷毀。")
    print(f"學術級拓撲數據報告路徑: {logger_service.csv_path}")
    print("==================================================")


# === 4. 實驗主入口 ===
if __name__ == "__main__":
    # 初始化共用儲存服務（腳本名稱設定為包含 authority 標記）
    logger_service = SimulationLoggerService(
        topic=TOPIC, model_name=GENERATION_MODEL, script_name="social-simulate-authority-topology"
    )

    # 執行階段一：生成 25 次對話
    execute_generation_phase()

    # 中間過渡：完全卸載 8B 模型並冷卻硬體
    unload_ollama_model(GENERATION_MODEL)

    # 執行階段二：載入大模型進行盲測與拓撲/瀑布指標計算
    execute_analysis_phase(logger_service)


>>> 進入階段一：對話模擬生成階段 (使用模型: qwen2.5:7b) <<<

Running Dialog Simulation for Run #1/25
✨ [Authority Selected] Agent_6 has been upgraded to AUTHORITY (Distinguished Research Fellow at Academia Sinica (Expert in Cyber Security & Digital Transformation))
--- Round 1/5 ---


C:\Users\TUF\AppData\Local\Temp\ipykernel_27492\2544730222.py:78: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=GENERATION_MODEL, temperature=0.7)


Agent_1 (Oppose): 尊敬的各位，

我在此就數位身分證（eID）的推行，分享我的一些看法。

首先，我想提及的是，正如劉老師所指出的，隱私保...
Agent_2 (Neutral): 尊敬的各位，

我閱讀了Agent_1分享的观点后，必须承认他对隐私保护和公众沟通的重要性给予了充分的关注。不过，在此我...
Agent_3 (Oppose): 尊敬的各位，

感謝Agent_1和Agent_2分享了各自的觀點，我深感這些討論對我們理解這個問題至關重要。

首先，...
Agent_4 (Neutral): 尊敬的各位，

感謝Agent_1和Agent_2分享了他們對數位身分證（eID）推行的看法，這確實為我們提供了一些有益...
Agent_5 (Support): 尊敬的各位，

我閱讀了幾位同仁對數位身分證（eID）推行的看法，確實引人深思。我想先表達自己的一些看法。

首先，我要...
[👑 AUTHORITY] Agent_6 (Support): 尊敬的各位，

我閱讀了Agent_1、Agent_2、Agent_3以及Agent_4提出的觀點，諸位分享的見解對此議...
--- Round 2/5 ---
Agent_1 (Oppose): 尊敬的各位，

首先，我對Agent_2、Agent_3和Agent_4分享的观点表示认同，并感谢你们从不同角度探讨了这...
Agent_2 (Neutral): 尊敬的各位，

首先，我對Agent_1、Agent_3和Agent_4分享的观点表示赞同，并感谢你们从不同角度探讨了这...
Agent_3 (Oppose): 尊敬的各位，

感謝所有同仁分享了他們對數位身分證（eID）推行的看法，這些觀點讓我受益匪淺。尤其是劉老師和Agent_...
Agent_4 (Neutral): 尊敬的各位，

感謝各位同仁分享了各自對數位身分證（eID）推行的看法，這讓我在思考這個問題時受益良多。

首先，我想再...
Agent_5 (Support): 尊敬的各位，

首先，我要感谢所有同仁分享了各自对数字身份证（eID）推行的看法，这为我提供了很多有价值的参考。特别感谢...
[👑 AUTHORITY] Agent_6 (Support): 尊敬的各位，

首先，我想对Agent_1、Agen